## 0. Setup

In [1]:
import os, sys
sys.path.append(os.getcwd())

from rag_utils import (
    get_api_key, get_embeddings, get_llm,
    load_pdfs, split_documents, build_vectorstore,
    get_retriever, build_rag_chain, ask, format_sources,
)

# Pick your provider: "gemini" (cloud, needs an API key) or "ollama" (local, needs `ollama serve` running)
PROVIDER = "ollama"  # <- change to "ollama" to run fully local

if PROVIDER == "gemini":
    get_api_key()
    print("Gemini API key configured.")
else:
    os.environ.setdefault("OLLAMA_CHAT_MODEL", "llama3.2")
    os.environ.setdefault("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
    print(f"Using local Ollama models: {os.environ['OLLAMA_CHAT_MODEL']} / {os.environ['OLLAMA_EMBEDDING_MODEL']}")
    print("Make sure `ollama serve` is running and both models are pulled.")


c:\Users\aa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using local Ollama models: llama3.2 / nomic-embed-text
Make sure `ollama serve` is running and both models are pulled.


## Part 1 — Test the RAG with Your Own CV



In [2]:
CV_PATH = "data/Mohamed_Abdallah_CV.pdf"  # <- change to your CV's path

cv_docs = load_pdfs([CV_PATH])
cv_chunks = split_documents(cv_docs, chunk_size=800, chunk_overlap=100)
print(f"Loaded {len(cv_docs)} pages -> {len(cv_chunks)} chunks")

embeddings = get_embeddings(provider=PROVIDER)
cv_vectordb = build_vectorstore(cv_chunks, embeddings, persist_directory="chroma_db/part1_cv", collection_name="cv_collection")
cv_retriever = get_retriever(cv_vectordb, k=4)

llm = get_llm(provider=PROVIDER)
cv_chain = build_rag_chain(llm)


Loaded 2 pages -> 7 chunks
Embedding chunks 1-7 of 7...


In [3]:
cv_questions = [
    "What are my main technical skills?",
    "What machine learning experience do I have?",
    "What programming languages do I know?",
    "What training or internships have I completed?",
]

for q in cv_questions:
    answer, docs = ask(cv_retriever, cv_chain, q)
    print(f"Q: {q}\nA: {answer}\n")
    print("Sources:")
    print(format_sources(docs))
    print("*" * 80)


Q: What are my main technical skills?
A: Based on the provided context, your main technical skills are:

1. Languages & Programming: Python, C++, SQL
2. AI / ML Frameworks: TensorFlow, Keras, Scikit-learn, OpenCV
3. Generative AI & LLMs: LLM APIs, Prompt Engineering, RAG, ChromaDB (Vector Database), NLP
4. Data & Databases: Pandas, NumPy, Matplotlib, SQL, Database Design
5. Tools & Version Control: Git, GitHub, Jupyter Notebook, VS Code

These are the main technical skills listed in the context.

Sources:
- Mohamed_Abdallah_CV.pdf (page 2)
********************************************************************************
Q: What machine learning experience do I have?
A: Based on the provided context, you have hands-on experience building Generative AI and LLM-based solutions, including a RAG chatbot using LLM prompting and vector search.

Sources:
- Mohamed_Abdallah_CV.pdf (page 1)
********************************************************************************
Q: What programming langua

## Part 2 — Ask Questions About a Book




In [4]:
BOOK_PATH = "data/[NLP] Natural Language Processing with PyTorch (2019).pdf"

book_docs = load_pdfs([BOOK_PATH])
print(f"Loaded {len(book_docs)} pages from the book")


Loaded 210 pages from the book


In [5]:
# --- Configuration A: chunk_size=800, chunk_overlap=100 ---
chunks_a = split_documents(book_docs, chunk_size=800, chunk_overlap=100)
vectordb_a = build_vectorstore(chunks_a, embeddings, persist_directory="chroma_db/part2_book_a", collection_name="book_a")
retriever_a = get_retriever(vectordb_a, k=4)

# --- Configuration B: chunk_size=500 (overlap fixed at 100) ---
chunks_b = split_documents(book_docs, chunk_size=500, chunk_overlap=100)
vectordb_b = build_vectorstore(chunks_b, embeddings, persist_directory="chroma_db/part2_book_b", collection_name="book_b")
retriever_b = get_retriever(vectordb_b, k=4)

print(f"Config A -> {len(chunks_a)} chunks (size=800, overlap=100)")
print(f"Config B -> {len(chunks_b)} chunks (size=500, overlap=100)")


Embedding chunks 1-50 of 711...
Embedding chunks 51-100 of 711...
Embedding chunks 101-150 of 711...
Embedding chunks 151-200 of 711...
Embedding chunks 201-250 of 711...
Embedding chunks 251-300 of 711...
Embedding chunks 301-350 of 711...
Embedding chunks 351-400 of 711...
Embedding chunks 401-450 of 711...
Embedding chunks 451-500 of 711...
Embedding chunks 501-550 of 711...
Embedding chunks 551-600 of 711...
Embedding chunks 601-650 of 711...
Embedding chunks 651-700 of 711...
Embedding chunks 701-711 of 711...
Embedding chunks 1-50 of 1161...
Embedding chunks 51-100 of 1161...
Embedding chunks 101-150 of 1161...
Embedding chunks 151-200 of 1161...
Embedding chunks 201-250 of 1161...
Embedding chunks 251-300 of 1161...
Embedding chunks 301-350 of 1161...
Embedding chunks 351-400 of 1161...
Embedding chunks 401-450 of 1161...
Embedding chunks 451-500 of 1161...
Embedding chunks 501-550 of 1161...
Embedding chunks 551-600 of 1161...
Embedding chunks 601-650 of 1161...
Embedding chunk

In [6]:
# Compare retrieved chunks for the SAME question under both configurations
compare_question = "What is the main argument of the second chapter?"  # <- adjust to your book

print("### Config A (chunk_size=800) retrieved chunks ###")
for d in retriever_a.invoke(compare_question):
    print(f"[p.{d.metadata.get('page')}] {d.page_content[:200]}...\n")

print("\n### Config B (chunk_size=500) retrieved chunks ###")
for d in retriever_b.invoke(compare_question):
    print(f"[p.{d.metadata.get('page')}] {d.page_content[:200]}...\n")


### Config A (chunk_size=800) retrieved chunks ###
[p.197] covered many NLP tasks in the remaining chapters, but here we briefly mention some important
topics that we could not address either in part or whole, due to limiting our scope to an initial
expositor...

[p.197] covered many NLP tasks in the remaining chapters, but here we briefly mention some important
topics that we could not address either in part or whole, due to limiting our scope to an initial
expositor...

[p.9] explain the supervised learning paradigm that will become the foundation for the book. If you are not
familiar with many of these terms so far, you’re in the right place. This chapter, along with futu...

[p.9] explain the supervised learning paradigm that will become the foundation for the book. If you are not
familiar with many of these terms so far, you’re in the right place. This chapter, along with futu...


### Config B (chunk_size=500) retrieved chunks ###
[p.196] Chapter 9. Classics, Frontiers, and Next


In [7]:
book_chain = build_rag_chain(llm)

book_questions = [
    "What topics does the book cover related to NLP?",
    "What is discussed on the first few pages about dialogue systems?",
]
unanswerable_question = "What is the capital of the fictional planet Zorblax?"

for q in book_questions + [unanswerable_question]:
    answer, docs = ask(retriever_a, book_chain, q)
    print(f"Q: {q}\nA: {answer}\n")
    print(format_sources(docs))
    print("-" * 80)


Q: What topics does the book cover related to NLP?
A: Based on the provided context, the book covers important topics in NLP, but the exact topics are not explicitly listed. However, it mentions that the book aims to bring newcomers to NLP and deep learning to a "tasting table" covering important topics in both areas.

- [NLP] Natural Language Processing with PyTorch (2019).pdf (page 198)
- [NLP] Natural Language Processing with PyTorch (2019).pdf (page 5)
--------------------------------------------------------------------------------
Q: What is discussed on the first few pages about dialogue systems?
A: Based on the provided context, on the first few pages, dialogue systems are discussed as a "holy grail" of computing, inspired by the Turing test and the Loebner Prize. They are also associated with artificial intelligence and popularized in pop culture by fictional systems like HAL 9000 in the film 2001: A Space Odyssey and the main computer on board the USS Enterprise in Star Trek.


## Part 3 — Multi-PDF RAG

Upload **3+ PDFs dynamically** (no hard-coded filename), combine them into one
ChromaDB collection, and ask cross-document questions. Each answer shows the
**source PDF filename and page number** it came from.


In [5]:
embeddings = get_embeddings(provider=PROVIDER)
llm = get_llm(provider=PROVIDER)

multi_docs = load_pdfs(PDF_PATHS)
...

Ellipsis

In [8]:
PDF_PATHS = [
    "data/Module3_Session1_Deck.pptx.pdf",  # <- replace with your 3+ PDFs
    "data/Module3_Session2_Deck.pptx.pdf",
    "data/Mohamed_Abdallah_CV.pdf",
]

multi_docs = load_pdfs(PDF_PATHS)
multi_chunks = split_documents(multi_docs, chunk_size=800, chunk_overlap=100)
print(f"Loaded {len(multi_docs)} pages from {len(PDF_PATHS)} PDFs -> {len(multi_chunks)} chunks")

multi_vectordb = build_vectorstore(multi_chunks, embeddings, persist_directory="chroma_db/part3_multi", collection_name="multi")
multi_retriever = get_retriever(multi_vectordb, k=4)
multi_chain = build_rag_chain(llm)


Loaded 50 pages from 3 PDFs -> 62 chunks
Embedding chunks 1-50 of 62...
Embedding chunks 51-62 of 62...


In [11]:
# Adjust these to match your actual PDFs' content and filenames
multi_test_questions = [
    "What is the KV-cache and why does it speed up generation?",  # Session 1 فقط
    "What is the causal mask and what does it block?" , # Session 2 فقط
    "What is a hallucination according to the definition given in the session?", 
    "What are my main technical skills?",
    "What machine learning experience do I have?",
    
]

for q in multi_test_questions:
    answer, docs = ask(multi_retriever, multi_chain, q)
    print(f"Q: {q}\nA: {answer}\n")
    print("Sources:")
    print(format_sources(docs))
    print("=" * 80)


Q: What is the KV-cache and why does it speed up generation?
A: According to the context, the KV-cache is a mechanism that speeds up generation by storing the computed Keys (K) and Values (V) for past tokens only once and reusing them for subsequent steps. This way, each new step only needs to process the newest token, attending over cached K/V, rather than re-computing attention Keys and Values for all previous tokens.

The KV-cache speeds up generation by reducing the computational overhead of re-computing attention Keys and Values for all previous tokens, which is a waste of computation.

Sources:
- Module3_Session1_Deck.pptx.pdf (page 17)
- Module3_Session1_Deck.pptx.pdf (page 22)
Q: What is the causal mask and what does it block?
A: According to the context, the causal mask is a mechanism that enforces the Autoregressive rule. It blocks future positions, meaning that a token can only see what has come before it, not what will come after.

Sources:
- Module3_Session1_Deck.pptx.pdf 